In [1]:
"""
Multi-Hierarchical RL Portfolio System - Data Preparation (Part 1 - v2)
========================================================================

Prepares SEPARATE data streams for:
1. Technical Agent: Price, volume, technical indicators
2. Sentiment Agent: News sentiment, social media, market sentiment

Key Features:
- No data leakage (proper rolling calculations)
- Handles tickers with different start dates (align to common period)
- Split-of-concept: technical and sentiment are completely separate
- Weekly frequency for rebalancing decisions
- Proper train/val/test splits
- Complete technical indicator suite matching specification

Portfolio: NVDA, MU, AAPL, AMD, ASML, MSFT, GOOG
"""

import pandas as pd
import numpy as np
import yfinance as yf
from pathlib import Path
import json
from datetime import datetime, timedelta
import warnings
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings('ignore')


class HierarchicalDataPreparator:
    """Prepares data for hierarchical RL system with complete technical indicators."""
    
    def __init__(
        self,
        tickers: list,
        start_date: str = '2020-01-01',
        end_date: str = None,
        train_split: float = 0.6,
        val_split: float = 0.2,
        output_dir: str = 'data_hierarchical',
        benchmark: str = 'QQQ',
    ):
        """
        Initialize data preparator for hierarchical RL system.
        
        Parameters
        ----------
        tickers : list
            List of stock tickers to include in portfolio
        start_date : str
            Start date for data collection
        end_date : str, optional
            End date (default: today)
        train_split : float
            Proportion of data for training (0.6 = 60%)
        val_split : float
            Proportion of data for validation (0.2 = 20%)
        output_dir : str
            Directory to save prepared data
        benchmark : str
            Benchmark ticker (e.g., SPY, QQQ)
        """
        self.tickers = [t.upper() for t in tickers]
        self.start_date = pd.to_datetime(start_date)
        self.end_date = pd.to_datetime(end_date) if end_date else pd.Timestamp.now()
        self.train_split = train_split
        self.val_split = val_split
        self.benchmark = benchmark
        self.output_dir = Path(output_dir)
        
        # Create directories
        self.output_dir.mkdir(exist_ok=True)
        (self.output_dir / 'technical').mkdir(exist_ok=True)
        (self.output_dir / 'sentiment').mkdir(exist_ok=True)
        
        print("="*70)
        print("HIERARCHICAL RL PORTFOLIO SYSTEM - DATA PREPARATION V2")
        print("="*70)
        print(f"Portfolio tickers: {', '.join(self.tickers)}")
        print(f"Benchmark: {self.benchmark}")
        print(f"Date range: {self.start_date.date()} to {self.end_date.date()}")
        print(f"Splits: Train={train_split:.0%}, Val={val_split:.0%}, Test={1-train_split-val_split:.0%}")
        print(f"Output: {self.output_dir}")
        print("="*70 + "\n")
        
    def prepare_all(self):
        """Run complete data preparation pipeline."""
        
        # Step 1: Download and align price data
        print("\n[1/5] Downloading and aligning price data...")
        weekly_prices = self._download_and_align_weekly()
        
        # Step 2: Prepare technical features (for Technical Agent)
        print("\n[2/5] Creating technical features...")
        technical_data = self._prepare_technical_features(weekly_prices)
        
        # Step 3: Prepare sentiment features (for Sentiment Agent)
        print("\n[3/5] Creating sentiment features...")
        sentiment_data = self._prepare_sentiment_features(weekly_prices)
        
        # Step 4: Create splits
        print("\n[4/5] Creating train/val/test splits...")
        splits = self._create_splits(technical_data, sentiment_data, weekly_prices)
        
        # Step 5: Save everything
        print("\n[5/5] Saving datasets...")
        self._save_all(splits)
        
        print("\n" + "="*70)
        print("DATA PREPARATION COMPLETE")
        print("="*70)
        print(f"Technical data: {self.output_dir / 'technical'}")
        print(f"Sentiment data: {self.output_dir / 'sentiment'}")
        print(f"Metadata: {self.output_dir / 'metadata.json'}")
        print("="*70 + "\n")
        
        return splits
    
    def _download_and_align_weekly(self):
        """Download data and align all tickers to common date range."""
        
        all_data = {}
        ticker_ranges = {}
        
        for ticker in self.tickers + [self.benchmark]:
            print(f"  Downloading {ticker}...")
            try:
                df = yf.download(
                    ticker,
                    start=self.start_date - timedelta(days=365),
                    end=self.end_date,
                    progress=False
                )
                
                if df.empty:
                    print(f"    Warning: No data available")
                    continue
                
                if isinstance(df.columns, pd.MultiIndex):
                    df.columns = df.columns.get_level_values(0)
                
                # Resample to weekly (Friday close)
                weekly = pd.DataFrame()
                weekly['open'] = df['Open'].resample('W-FRI').first()
                weekly['high'] = df['High'].resample('W-FRI').max()
                weekly['low'] = df['Low'].resample('W-FRI').min()
                weekly['close'] = df['Adj Close' if 'Adj Close' in df.columns else 'Close'].resample('W-FRI').last()
                weekly['volume'] = df['Volume'].resample('W-FRI').sum()
                weekly['return'] = np.log(weekly['close'] / weekly['close'].shift(1))
                weekly = weekly.dropna()
                
                all_data[ticker] = weekly
                ticker_ranges[ticker] = {
                    'start': weekly.index.min(),
                    'end': weekly.index.max(),
                    'weeks': len(weekly)
                }
                
                print(f"    Downloaded {len(weekly)} weeks from {weekly.index.min().date()} to {weekly.index.max().date()}")
                
            except Exception as e:
                print(f"    Error: {e}")
                continue
        
        if not all_data:
            raise ValueError("No data successfully downloaded!")
        
        # Find common date range
        common_start = max(info['start'] for info in ticker_ranges.values())
        common_end = min(info['end'] for info in ticker_ranges.values())
        
        print(f"\n  Common coverage: {common_start.date()} to {common_end.date()}")
        
        # Align all tickers
        aligned_data = {}
        for ticker, df in all_data.items():
            aligned = df[(df.index >= common_start) & (df.index <= common_end)]
            aligned_data[ticker] = aligned
            print(f"    {ticker}: {len(aligned)} weeks")
        
        lengths = [len(df) for df in aligned_data.values()]
        if len(set(lengths)) > 1:
            print(f"  Warning: Different lengths after alignment: {set(lengths)}")
        else:
            print(f"  All tickers aligned: {lengths[0]} weeks")
        
        return aligned_data
    
    def _prepare_technical_features(self, weekly_prices):
        """
        Create technical features for Technical Agent.
        
        Features match specification exactly:
        - Trend: SMA (4,8,12w), EMA (8,12w), MACD
        - Momentum: Return lags (1,2,3w), RSI (14w), Stochastic (14w), ROC (4w)
        - Volatility: Historical vol (12w), ATR (14w), Bollinger Bands (20w)
        - Volume: Volume ratio (20w), MFI (14w), OBV ROC (4w)
        - Benchmark: Correlation (12w), Beta (12w)
        """
        technical_features = []
        
        bench_df = weekly_prices.get(self.benchmark)
        
        for ticker in self.tickers:
            if ticker not in weekly_prices:
                continue
                
            print(f"    Processing {ticker}...")
            df = weekly_prices[ticker].copy()
            close = df['close']
            high = df['high']
            low = df['low']
            volume = df['volume']
            returns = df['return']
            
            # === TREND INDICATORS ===
            
            # Simple Moving Averages (4, 8, 12 weeks)
            for w in [4, 8, 12]:
                sma = close.rolling(w).mean()
                df[f'price_to_sma_{w}w'] = close / sma
            
            # Exponential Moving Averages (8, 12 weeks)
            for w in [8, 12]:
                ema = close.ewm(span=w, adjust=False).mean()
                df[f'price_to_ema_{w}w'] = close / ema
            
            # MACD Histogram (12-week fast, 26-week slow, 9-week signal)
            ema_fast = close.ewm(span=12, adjust=False).mean()
            ema_slow = close.ewm(span=26, adjust=False).mean()
            macd_line = ema_fast - ema_slow
            signal_line = macd_line.ewm(span=9, adjust=False).mean()
            df['macd_histogram'] = macd_line - signal_line
            # Normalize by price
            df['macd_histogram'] = df['macd_histogram'] / close
            
            # === MOMENTUM INDICATORS ===
            
            # Return Lags (1, 2, 3 weeks)
            for lag in [1, 2, 3]:
                df[f'return_lag_{lag}w'] = returns.shift(lag)
            
            # RSI (14-week)
            delta = close.diff()
            gain = delta.where(delta > 0, 0).rolling(14).mean()
            loss = -delta.where(delta < 0, 0).rolling(14).mean()
            rs = gain / (loss + 1e-10)
            df['rsi_14w'] = 100 - (100 / (1 + rs))
            
            # Stochastic Oscillator (14-week %K)
            lowest_low = low.rolling(14).min()
            highest_high = high.rolling(14).max()
            df['stochastic_14w'] = 100 * (close - lowest_low) / (highest_high - lowest_low + 1e-10)
            
            # Rate of Change (4-week)
            df['roc_4w'] = close.pct_change(4) * 100
            
            # === VOLATILITY INDICATORS ===
            
            # Historical Volatility (12-week rolling std, annualized)
            df['volatility_12w'] = returns.rolling(12).std() * np.sqrt(52)
            
            # ATR Percentage (14-week)
            prev_close = close.shift(1)
            tr = pd.concat([
                high - low,
                (high - prev_close).abs(),
                (low - prev_close).abs()
            ], axis=1).max(axis=1)
            atr = tr.ewm(span=14, adjust=False).mean()
            df['atr_pct_14w'] = (atr / close) * 100
            
            # Bollinger Band Position (20-week, 2 std dev)
            sma_20 = close.rolling(20).mean()
            std_20 = close.rolling(20).std()
            upper_band = sma_20 + (2 * std_20)
            lower_band = sma_20 - (2 * std_20)
            # Position within bands: 0 = lower, 0.5 = middle, 1 = upper
            df['bb_position_20w'] = (close - lower_band) / (upper_band - lower_band + 1e-10)
            
            # === VOLUME INDICATORS ===
            
            # Volume Ratio (current vs 20-week average)
            vol_ma_20 = volume.rolling(20).mean()
            df['volume_ratio_20w'] = volume / (vol_ma_20 + 1e-10)
            
            # Money Flow Index (14-week volume-weighted momentum)
            typical_price = (high + low + close) / 3
            raw_money_flow = typical_price * volume
            
            positive_flow = raw_money_flow.where(typical_price > typical_price.shift(1), 0)
            negative_flow = raw_money_flow.where(typical_price < typical_price.shift(1), 0)
            
            positive_mf = positive_flow.rolling(14).sum()
            negative_mf = negative_flow.rolling(14).sum()
            
            mfi_ratio = positive_mf / (negative_mf + 1e-10)
            df['mfi_14w'] = 100 - (100 / (1 + mfi_ratio))
            
            # OBV Rate of Change (4-week)
            obv = (np.sign(close.diff()) * volume).fillna(0).cumsum()
            df['obv_roc_4w'] = obv.pct_change(4) * 100
            
            # === BENCHMARK RELATIVE METRICS ===
            
            if bench_df is not None:
                common_dates = df.index.intersection(bench_df.index)
                if len(common_dates) > 12:
                    asset_rets = returns.loc[common_dates]
                    bench_rets = bench_df.loc[common_dates, 'return']
                    
                    # Rolling correlation (12-week, no look-ahead)
                    corr_vals = []
                    for i in range(len(common_dates)):
                        if i < 11:
                            corr_vals.append(np.nan)
                        else:
                            window_asset = asset_rets.iloc[i-11:i+1]
                            window_bench = bench_rets.iloc[i-11:i+1]
                            corr_vals.append(window_asset.corr(window_bench))
                    
                    df.loc[common_dates, 'bench_corr_12w'] = corr_vals
                    
                    # Rolling beta (12-week, no look-ahead)
                    beta_vals = []
                    for i in range(len(common_dates)):
                        if i < 11:
                            beta_vals.append(np.nan)
                        else:
                            window_asset = asset_rets.iloc[i-11:i+1]
                            window_bench = bench_rets.iloc[i-11:i+1]
                            cov = window_asset.cov(window_bench)
                            var = window_bench.var()
                            beta_vals.append(cov / (var + 1e-8))
                    
                    df.loc[common_dates, 'bench_beta_12w'] = beta_vals
            
            # Add ticker identifier
            df['ticker'] = ticker
            
            # Drop rows with NaN (from rolling calculations)
            df = df.dropna()
            
            technical_features.append(df)
        
        # Combine all tickers
        technical_df = pd.concat(technical_features, axis=0).sort_index()
        
        print(f"\n  Technical features created: {technical_df.shape}")
        feature_cols = [c for c in technical_df.columns if c not in ['ticker', 'open', 'high', 'low', 'close', 'volume', 'return']]
        print(f"    Feature count: {len(feature_cols)}")
        print(f"    Features: {feature_cols}")
        
        return technical_df
    
    def _prepare_sentiment_features(self, weekly_prices):
        """
        Create sentiment features for Sentiment Agent.
        
        Features:
        - Market sentiment (VIX, credit spreads)
        - Price-based sentiment proxies
        - Volume-based sentiment
        - Risk sentiment
        """
        sentiment_features = []
        
        # Download market sentiment indicators
        print("    Downloading market sentiment indicators...")
        sentiment_indicators = self._get_market_sentiment()
        
        for ticker in self.tickers:
            if ticker not in weekly_prices:
                continue
                
            print(f"    Processing {ticker}...")
            df = weekly_prices[ticker].copy()
            returns = df['return']
            volume = df['volume']
            close = df['close']
            
            # === PRICE-BASED SENTIMENT PROXIES ===
            
            # Momentum sentiment (rolling returns)
            for w in [2, 4, 6]:
                df[f'momentum_{w}w'] = returns.rolling(w).sum()
            
            # Trend strength (% of positive weeks)
            df['positive_weeks_pct'] = (returns > 0).rolling(6).mean()
            
            # Price dispersion (high/low range)
            df['price_dispersion'] = (df['high'] - df['low']) / close
            
            # === VOLUME-BASED SENTIMENT ===
            
            # Volume trend (increasing = bullish)
            vol_sma_short = volume.rolling(2).mean()
            vol_sma_long = volume.rolling(6).mean()
            df['volume_sentiment'] = (vol_sma_short / vol_sma_long) - 1
            
            # Volume surge (compared to average)
            vol_std = volume.rolling(8).std()
            vol_mean = volume.rolling(8).mean()
            df['volume_surge'] = (volume - vol_mean) / (vol_std + 1e-10)
            
            # === RISK SENTIMENT ===
            
            # Volatility regime (normalized)
            vol = returns.rolling(4).std()
            vol_ma = vol.rolling(8).mean()
            vol_std = vol.rolling(8).std()
            df['vol_regime'] = (vol - vol_ma) / (vol_std + 1e-8)
            
            # Drawdown (from recent high)
            rolling_max = close.rolling(6).max()
            df['drawdown'] = (close - rolling_max) / rolling_max
            
            # === MARKET SENTIMENT ===
            if sentiment_indicators is not None:
                common_dates = df.index.intersection(sentiment_indicators.index)
                for col in sentiment_indicators.columns:
                    df.loc[common_dates, col] = sentiment_indicators.loc[common_dates, col]
            
            # === SIMULATED NEWS SENTIMENT ===
            df['news_sentiment'] = self._simulate_news_sentiment(returns)
            
            # Add ticker identifier
            df['ticker'] = ticker
            
            # Drop NaN rows
            df = df.dropna()
            
            sentiment_features.append(df)
        
        # Combine all tickers
        sentiment_df = pd.concat(sentiment_features, axis=0).sort_index()
        
        print(f"\n  Sentiment features created: {sentiment_df.shape}")
        feature_cols = [c for c in sentiment_df.columns if c not in ['ticker', 'open', 'high', 'low', 'close', 'volume', 'return']]
        print(f"    Feature count: {len(feature_cols)}")
        
        return sentiment_df
    
    def _get_market_sentiment(self):
        """Get market-wide sentiment indicators."""
        try:
            # VIX (fear index)
            vix = yf.download('^VIX', start=self.start_date - timedelta(days=365), 
                            end=self.end_date, progress=False)
            if not vix.empty:
                if isinstance(vix.columns, pd.MultiIndex):
                    vix.columns = vix.columns.get_level_values(0)
                vix_weekly = vix['Close'].resample('W-FRI').last()
                
                # Normalize VIX
                vix_ma = vix_weekly.rolling(26).mean()
                vix_std = vix_weekly.rolling(26).std()
                market_fear = (vix_weekly - vix_ma) / (vix_std + 1e-8)
                
                df = pd.DataFrame({'market_fear': market_fear})
                
                # High yield spread (use TLT/HYG ratio as proxy)
                try:
                    tlt = yf.download('TLT', start=self.start_date - timedelta(days=365),
                                    end=self.end_date, progress=False)
                    hyg = yf.download('HYG', start=self.start_date - timedelta(days=365),
                                    end=self.end_date, progress=False)
                    
                    if not tlt.empty and not hyg.empty:
                        if isinstance(tlt.columns, pd.MultiIndex):
                            tlt.columns = tlt.columns.get_level_values(0)
                        if isinstance(hyg.columns, pd.MultiIndex):
                            hyg.columns = hyg.columns.get_level_values(0)
                            
                        tlt_weekly = tlt['Close'].resample('W-FRI').last()
                        hyg_weekly = hyg['Close'].resample('W-FRI').last()
                        
                        spread = (tlt_weekly / hyg_weekly).pct_change(4)
                        df['credit_sentiment'] = spread
                except:
                    pass
                
                return df.dropna()
                
        except Exception as e:
            print(f"    Warning: Could not download sentiment indicators: {e}")
            return None
    
    def _simulate_news_sentiment(self, returns, noise_level=0.3):
        """
        Simulate news sentiment from returns.
        In production, replace with real sentiment from NewsAPI/Twitter API.
        """
        sentiment_base = returns.rolling(2).mean()
        sentiment_momentum = returns.rolling(4).mean()
        
        noise = np.random.normal(0, noise_level, len(returns))
        
        raw_sentiment = 0.5 * sentiment_base + 0.3 * sentiment_momentum + noise
        bounded_sentiment = 2 / (1 + np.exp(-raw_sentiment)) - 1
        
        return pd.Series(bounded_sentiment, index=returns.index)
    
    def _normalize_features(self, splits):
        """
        Normalize features using StandardScaler.
        Fit on training data only to prevent data leakage.
        Skip binary features (e.g., 0/1 indicators).
        """
        # Columns to exclude from normalization
        base_cols = ['ticker', 'open', 'high', 'low', 'close', 'volume', 'return']
        
        # Normalize technical features
        tech_train = splits['train']['technical']
        tech_feature_cols = [c for c in tech_train.columns if c not in base_cols]
        
        if tech_feature_cols:
            # Identify binary features (only 2 unique values)
            binary_tech_features = []
            for col in tech_feature_cols:
                unique_vals = tech_train[col].nunique()
                if unique_vals <= 2:
                    binary_tech_features.append(col)
            
            # Features to normalize (exclude binary)
            tech_normalize_cols = [c for c in tech_feature_cols if c not in binary_tech_features]
            
            if binary_tech_features:
                print(f"    Technical: Skipping binary features: {binary_tech_features}")
            
            if tech_normalize_cols:
                print(f"    Technical: Normalizing {len(tech_normalize_cols)} features")
                
                # Fit scaler on training data
                tech_scaler = StandardScaler()
                tech_scaler.fit(tech_train[tech_normalize_cols])
                
                # Transform all splits
                for split_name in ['train', 'val', 'test']:
                    normalized_values = tech_scaler.transform(
                        splits[split_name]['technical'][tech_normalize_cols]
                    )
                    splits[split_name]['technical'][tech_normalize_cols] = normalized_values
                
                # Store scaler info for metadata
                self.tech_scaler_mean = tech_scaler.mean_.tolist()
                self.tech_scaler_std = tech_scaler.scale_.tolist()
                self.tech_normalized_cols = tech_normalize_cols
                self.tech_binary_cols = binary_tech_features
            else:
                print(f"    Technical: No features to normalize")
                self.tech_scaler_mean = []
                self.tech_scaler_std = []
                self.tech_normalized_cols = []
                self.tech_binary_cols = binary_tech_features
        
        # Normalize sentiment features
        sent_train = splits['train']['sentiment']
        sent_feature_cols = [c for c in sent_train.columns if c not in base_cols]
        
        if sent_feature_cols:
            # Identify binary features
            binary_sent_features = []
            for col in sent_feature_cols:
                unique_vals = sent_train[col].nunique()
                if unique_vals <= 2:
                    binary_sent_features.append(col)
            
            # Features to normalize (exclude binary)
            sent_normalize_cols = [c for c in sent_feature_cols if c not in binary_sent_features]
            
            if binary_sent_features:
                print(f"    Sentiment: Skipping binary features: {binary_sent_features}")
            
            if sent_normalize_cols:
                print(f"    Sentiment: Normalizing {len(sent_normalize_cols)} features")
                
                # Fit scaler on training data
                sent_scaler = StandardScaler()
                sent_scaler.fit(sent_train[sent_normalize_cols])
                
                # Transform all splits
                for split_name in ['train', 'val', 'test']:
                    normalized_values = sent_scaler.transform(
                        splits[split_name]['sentiment'][sent_normalize_cols]
                    )
                    splits[split_name]['sentiment'][sent_normalize_cols] = normalized_values
                
                # Store scaler info for metadata
                self.sent_scaler_mean = sent_scaler.mean_.tolist()
                self.sent_scaler_std = sent_scaler.scale_.tolist()
                self.sent_normalized_cols = sent_normalize_cols
                self.sent_binary_cols = binary_sent_features
            else:
                print(f"    Sentiment: No features to normalize")
                self.sent_scaler_mean = []
                self.sent_scaler_std = []
                self.sent_normalized_cols = []
                self.sent_binary_cols = binary_sent_features
        
        return splits
    
    def _create_splits(self, technical_df, sentiment_df, weekly_prices):
        """Create train/val/test splits with normalization."""
        
        all_dates = sorted(technical_df.index.unique())
        n_dates = len(all_dates)
        
        if n_dates < 3:
            raise ValueError(f"Not enough data for splits. Only {n_dates} weeks available.")
        
        train_end_idx = int(n_dates * self.train_split)
        val_end_idx = int(n_dates * (self.train_split + self.val_split))
        
        train_end_idx = min(train_end_idx, n_dates - 2)
        val_end_idx = min(val_end_idx, n_dates - 1)
        
        train_end = all_dates[train_end_idx]
        val_end = all_dates[val_end_idx]
        
        print(f"  Train: {all_dates[0].date()} to {train_end.date()} ({train_end_idx} weeks)")
        print(f"  Val:   {all_dates[train_end_idx+1].date()} to {val_end.date()} ({val_end_idx - train_end_idx} weeks)")
        print(f"  Test:  {all_dates[val_end_idx+1].date()} to {all_dates[-1].date()} ({n_dates - val_end_idx - 1} weeks)")
        
        splits = {}
        
        for split_name, start_idx, end_idx in [
            ('train', 0, train_end_idx),
            ('val', train_end_idx + 1, val_end_idx),
            ('test', val_end_idx + 1, n_dates)
        ]:
            split_dates = all_dates[start_idx:end_idx]
            
            tech_split = technical_df[technical_df.index.isin(split_dates)].copy()
            sent_split = sentiment_df[sentiment_df.index.isin(split_dates)].copy()
            
            returns_data = []
            for ticker in self.tickers:
                if ticker in weekly_prices:
                    ret_df = weekly_prices[ticker][['return']].copy()
                    ret_df = ret_df[ret_df.index.isin(split_dates)]
                    ret_df.columns = [ticker]
                    returns_data.append(ret_df)
            
            returns_split = pd.concat(returns_data, axis=1)
            
            splits[split_name] = {
                'technical': tech_split,
                'sentiment': sent_split,
                'returns': returns_split
            }
            
            print(f"    {split_name}: tech={tech_split.shape}, sent={sent_split.shape}, returns={returns_split.shape}")
        
        # Normalize features after splitting to prevent data leakage
        print("\n  Normalizing features...")
        splits = self._normalize_features(splits)
        
        return splits
    
    def _save_all(self, splits):
        """Save all datasets and metadata."""
        
        for split_name, data in splits.items():
            tech_path = self.output_dir / 'technical' / f'{split_name}.csv'
            data['technical'].round(4).to_csv(tech_path)
            
            sent_path = self.output_dir / 'sentiment' / f'{split_name}.csv'
            data['sentiment'].round(4).to_csv(sent_path)
            
            ret_path = self.output_dir / f'returns_{split_name}.csv'
            data['returns'].round(4).to_csv(ret_path)
            
            print(f"  Saved {split_name} split")
        
        # Metadata
        # Separate indicator features (for RL) from base columns
        base_cols = ['open', 'high', 'low', 'close', 'volume', 'return']
        tech_indicator_cols = [c for c in splits['train']['technical'].columns 
                               if c not in base_cols + ['ticker']]
        sent_indicator_cols = [c for c in splits['train']['sentiment'].columns 
                               if c not in base_cols + ['ticker']]
        
        metadata = {
            'tickers': self.tickers,
            'benchmark': self.benchmark,
            'date_range': {
                'start': self.start_date.strftime('%Y-%m-%d'),
                'end': self.end_date.strftime('%Y-%m-%d')
            },
            'splits': {
                'train': self.train_split,
                'val': self.val_split,
                'test': 1 - self.train_split - self.val_split
            },
            'technical_features': [c for c in splits['train']['technical'].columns if c not in ['ticker']],
            'sentiment_features': [c for c in splits['train']['sentiment'].columns if c not in ['ticker']],
            'technical_indicator_features': tech_indicator_cols,
            'sentiment_indicator_features': sent_indicator_cols,
            'normalization': {
                'technical': {
                    'normalized_features': getattr(self, 'tech_normalized_cols', []),
                    'binary_features': getattr(self, 'tech_binary_cols', []),
                    'scaler_mean': getattr(self, 'tech_scaler_mean', []),
                    'scaler_std': getattr(self, 'tech_scaler_std', [])
                },
                'sentiment': {
                    'normalized_features': getattr(self, 'sent_normalized_cols', []),
                    'binary_features': getattr(self, 'sent_binary_cols', []),
                    'scaler_mean': getattr(self, 'sent_scaler_mean', []),
                    'scaler_std': getattr(self, 'sent_scaler_std', [])
                }
            },
            'created_at': datetime.now().isoformat()
        }
        
        with open(self.output_dir / 'metadata.json', 'w') as f:
            json.dump(metadata, f, indent=2)
        
        print(f"  Saved metadata with normalization info")


In [2]:


# Portfolio with established tickers (longer history)
PORTFOLIO = ['NVDA', 'MU', 'AAPL', 'AMD', 'ASML', 'MSFT', 'GOOG']

preparator = HierarchicalDataPreparator(
    tickers=PORTFOLIO,
    start_date='2020-01-01',
    end_date=None,
    train_split=0.6,
    val_split=0.2,
    output_dir='data_hierarchical',
    benchmark='QQQ'
)

splits = preparator.prepare_all()

print("\nData preparation complete!")
print("\nNext steps:")
print("1. Review data in data_hierarchical/")
print("2. Run train_part1_agents.ipynb to train agents")
print("3. Technical Agent uses: data_hierarchical/technical/")
print("4. Sentiment Agent uses: data_hierarchical/sentiment/")



HIERARCHICAL RL PORTFOLIO SYSTEM - DATA PREPARATION V2
Portfolio tickers: NVDA, MU, AAPL, AMD, ASML, MSFT, GOOG
Benchmark: QQQ
Date range: 2020-01-01 to 2025-10-24
Splits: Train=60%, Val=20%, Test=20%
Output: data_hierarchical


[1/5] Downloading and aligning price data...
    Downloaded 355 weeks from 2019-01-11 to 2025-10-24
    Downloaded 355 weeks from 2019-01-11 to 2025-10-24
    Downloaded 355 weeks from 2019-01-11 to 2025-10-24
    Downloaded 355 weeks from 2019-01-11 to 2025-10-24
    Downloaded 355 weeks from 2019-01-11 to 2025-10-24
    Downloaded 355 weeks from 2019-01-11 to 2025-10-24
    Downloaded 355 weeks from 2019-01-11 to 2025-10-24
    Downloaded 355 weeks from 2019-01-11 to 2025-10-24

  Common coverage: 2019-01-11 to 2025-10-24
    NVDA: 355 weeks
    MU: 355 weeks
    AAPL: 355 weeks
    AMD: 355 weeks
    ASML: 355 weeks
    MSFT: 355 weeks
    GOOG: 355 weeks
    QQQ: 355 weeks
  All tickers aligned: 355 weeks

[2/5] Creating technical features...
    Processing